In [2]:
!pip install ultralytics

In [53]:
from collections import defaultdict

import cv2
import numpy as np
from google.colab.patches import cv2_imshow
from ultralytics import YOLO
from google.colab import drive
drive.mount('/content/drive')

# Load the YOLOv8 model
model = YOLO('yolov8n.pt')

# Open the video file
video_path ='/content/drive/MyDrive/0_497Project/preliminarydata/results/clipped_street_crossing.mov'
output_path = '/content/drive/MyDrive/0_497Project/preliminarydata/results/tracking_objects2.csv'

# Open CSV file to store box positions
with open(output_path, 'w', newline='') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow(['Frame', 'Track ID', 'X', 'Y', 'Width', 'Height'])

    cap = cv2.VideoCapture(video_path)

    # Store the track history
    #defaultdict that stores the history of tracked objects
    track_history = defaultdict(lambda: [])

    # Loop through the video frames
    #It's a loop that processes each frame of the video until it reaches the end or the user presses
    while cap.isOpened():
        # Read a frame from the video
        success, frame = cap.read()

        if success:
            # Run YOLOv8 tracking on the frame, persisting tracks between frames
            #It retrieves the bounding boxes and track IDs of detected objects.
            results = model.track(frame, persist=True)
            if results[0].boxes.id != None:
              print(results[0].boxes)
              # Get the boxes and track IDs
              boxes = results[0].boxes.xywh.cpu()
              track_ids = results[0].boxes.id.int().cpu().tolist()

              # Visualize the results on the frame
              annotated_frame = results[0].plot()
              #save tracking history
              if len(track_ids)==1:
                csv_writer.writerow([cap.get(cv2.CAP_PROP_POS_FRAMES), track_ids[0],
                                      float(results[0].boxes.xywh[0][0]), float(results[0].boxes.xywh[0][1]),
                                      float(results[0].boxes.xywh[0][2]), float(results[0].boxes.xywh[0][3])])

              else:
                for i in range(len(track_ids)):
                  csv_writer.writerow([cap.get(cv2.CAP_PROP_POS_FRAMES), track_ids[i],
                                      float(results[0].boxes.xywh[i][0]), float(results[0].boxes.xywh[i][1]),
                                      float(results[0].boxes.xywh[i][2]), float(results[0].boxes.xywh[i][3])])

            else:
              continue

            # Plot the tracks
            for box, track_id in zip(boxes, track_ids):
                x, y, w, h = box
                track = track_history[track_id]
                track.append((float(x), float(y)))  # x, y center point
                if len(track) > 30:  # retain 90 tracks for 90 frames
                    track.pop(0)

                # Draw the tracking lines
                points = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                cv2.polylines(annotated_frame, [points], isClosed=False, color=(230, 230, 230), thickness=10)

            # Display the annotated frame
            #cv2_imshow( annotated_frame)#"YOLOv8 Tracking",

            # Break the loop if 'q' is pressed
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
        else:
            # Break the loop if the end of the video is reached
            break

    # Release the video capture object and close the display window
    cap.release()
    cv2.destroyAllWindows()
csv_file.close()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

0: 384x640 (no detections), 188.2ms
Speed: 5.0ms preprocess, 188.2ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 144.1ms
Speed: 2.5ms preprocess, 144.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 158.5ms
Speed: 6.0ms preprocess, 158.5ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 (no detections), 162.0ms
Speed: 4.8ms preprocess, 162.0ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 165.3ms
Speed: 2.8ms preprocess, 165.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 163.3ms
Speed: 5.0ms preprocess, 163.3ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 car, 154.9ms
Speed: 4.2ms preprocess, 154.9ms inference, 1.1ms post

# Find speed Trajectory

In [58]:
import pandas as pd
import math

# Define a function to calculate distance between two points
def distance(x1, y1, x2, y2):
    return math.sqrt((x2 - x1)**2 + (y2 - y1)**2)

# Read CSV file
csv_file_path = '/content/drive/MyDrive/0_497Project/preliminarydata/results/tracked_objects.csv'
# Read CSV file using pandas
df = pd.read_csv(csv_file_path)

# Initialize previous row variables
previous_frame = None
previous_track_id = None
previous_x = None
previous_y = None

# Initialize lists to store centroids and speeds
centroids = []
speeds = []

# Iterate over unique track IDs
for track_id in df['Track ID'].unique():
    # Filter DataFrame for current track ID
    track_df = df[df['Track ID'] == track_id]

    # Calculate speeds and centroids for the current track ID
    for index, row in track_df.iterrows():
        frame = row['Frame']
        x = row['X']
        y = row['Y']
        width = row['Width']
        height = row['Height']

        # Calculate centroid of the box
        centroid_x = x + width / 2
        centroid_y = y + height / 2

        # Append centroid to centroids list
        centroids.append((frame, track_id, centroid_x, centroid_y))

        # Calculate speed if there is a previous row for the same track ID
        if previous_frame is not None and track_id == previous_track_id:
            # Calculate distance between centroids
            dist = distance(centroid_x, centroid_y, previous_x, previous_y)

            # Calculate speed (distance / time)
            time_diff = frame - previous_frame
            speed = dist / time_diff if time_diff != 0 else 0

            # Append speed to speeds list
            speeds.append((frame, track_id, speed))
        else:
            speeds.append((frame, track_id, 0))

        # Update previous row variables
        previous_frame = frame
        previous_track_id = track_id
        previous_x = centroid_x
        previous_y = centroid_y

# Create DataFrame for centroids and speeds
centroids_df = pd.DataFrame(centroids, columns=['Frame', 'Track ID', 'Centroid X', 'Centroid Y'])
speeds_df = pd.DataFrame(speeds, columns=['Frame', 'Track ID', 'Speed'])

# Merge original DataFrame with centroids and speeds DataFrames
merged_df = pd.merge(df, centroids_df, on=['Frame', 'Track ID'], how='left')
merged_df = pd.merge(merged_df, speeds_df, on=['Frame', 'Track ID'], how='left')

# Save the merged DataFrame to a new CSV file
new_csv_file_path = '/content/drive/MyDrive/0_497Project/preliminarydata/results/merged_data.csv'
merged_df.to_csv(new_csv_file_path, index=False)

print("CSV file with merged data saved successfully.")


CSV file with merged data saved successfully.
